# Train the tobacco-leaf classifier on a Colab GPU
Run the cells **top to bottom**. Before you start: *Runtime → Change runtime type → T4 GPU*.
The dataset is downloaded from Hugging Face automatically. Nothing private is needed, so **do not type any keys or passwords** into this notebook.

In [ ]:
# 1. Is a GPU visible?
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

**2. Upload `ml_for_colab.zip`** (made on your PC with `python colab/make_zip.py` inside the `ml` folder). It holds code only.

In [ ]:
%cd /content
from google.colab import files
up = files.upload()            # choose ml_for_colab.zip (NOT make_zip.py)
assert "ml_for_colab.zip" in up, "Please upload the file named ml_for_colab.zip"
!rm -rf /content/ml && mkdir /content/ml && unzip -q -o ml_for_colab.zip -d /content/ml
%cd /content/ml
!pip -q install datasets scikit-learn pyyaml pillow
%env HF_HUB_DISABLE_PROGRESS_BARS=1

## 3. Tuning trials (validation only)
Each trial adds one idea to the previous one, so you can see what helped. They use `--skip-test`, so **the test set is not used to choose anything**.
I have not timed these on a GPU, so the first trial doubles as a speed check.

In [ ]:
COMMON = "--epochs 30 --lr 0.005 --cosine --skip-test"
!python -m src.training.centralised $COMMON --out r1_cosine

In [ ]:
!python -m src.training.centralised $COMMON --strong-aug --label-smoothing 0.1 --out r2_aug

In [ ]:
!python -m src.training.centralised $COMMON --strong-aug --label-smoothing 0.1 --balance --out r3_balance

In [ ]:
!python -m src.training.centralised $COMMON --strong-aug --label-smoothing 0.1 --balance --img-size 224 --out r4_224px

## 4. Compare on VALIDATION macro-F1 and pick one

In [ ]:
import json, pathlib
rows = []
for d in sorted(pathlib.Path("outputs").glob("r*")):
    r = json.loads((d / "result.json").read_text())
    h = r["history"]
    last5 = sum(e["valMacroF1"] for e in h[-5:]) / 5
    rows.append((d.name, r["bestValMacroF1"], last5, max(e["valAccuracy"] for e in h)))
print(f"{'run':14} {'best val F1':>12} {'mean of last 5':>15} {'best val acc':>13}")
for n, b, l, a in sorted(rows, key=lambda t: -t[1]):
    print(f"{n:14} {b:12.3f} {l:15.3f} {a:13.3f}")
print("Note: the last-5 average shows how stable training is; a high best score alone can be luck.")

## 5. Test the chosen run ONCE
Put the run's folder name below. The script refuses to run a second time on the same folder, so the test set stays honest.

In [ ]:
CHOSEN = "r4_224px"   # <- change to the run you picked above
!python -m src.training.final_test --out $CHOSEN

In [ ]:
# 6. Download the chosen model. Save it on your PC as ml/outputs/<CHOSEN>/model.pt and result.json
import shutil
shutil.make_archive("chosen_model", "zip", f"outputs/{CHOSEN}")
files.download("chosen_model.zip")